# Lab 1 essentials: read seven documents, cut them into pieces, and search them

**Time: about 50 minutes. Click Run all, then read each step while it runs.**

**What you will do.** FreshCart, a food delivery company, has seven support documents in four formats:
PDF, Word, PowerPoint and Excel. You will load them, cut them into small pieces, attach a label to
each piece, and search them with a real question. Along the way you will find five problems that no
error message tells you about.

**The libraries, and what each one does.** Nothing else is used: no lab-specific code.

| Library | What it does here | Step | Docs |
|---|---|---|---|
| `langchain-community` | ready-made **loaders** that read a file into LangChain `Document`s | 2, 3 | [python.langchain.com](https://python.langchain.com/docs/integrations/document_loaders/) |
| `pypdf` | reads the text of a PDF (used by LangChain's PDF loader) | 2 | [pypdf.readthedocs.io](https://pypdf.readthedocs.io/) |
| `docx2txt` | reads the text of a Word file (used by LangChain's Word loader) | 3 | [pypi.org/project/docx2txt](https://pypi.org/project/docx2txt/) |
| `python-docx` | reads a Word table cell by cell, to check what the loader lost | 3b | [python-docx.readthedocs.io](https://python-docx.readthedocs.io/) |
| `python-pptx` | reads the text of PowerPoint slides | 3 | [python-pptx.readthedocs.io](https://python-pptx.readthedocs.io/) |
| `pandas` + `openpyxl` | reads an Excel sheet into a table | 3, 4b | [pandas read_excel](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html) |
| `langchain-core` | the `Document` object (text plus labels) and a simple in-memory **vector store** | 3, 6, 7 | [python.langchain.com](https://python.langchain.com/docs/concepts/vectorstores/) |
| `langchain-text-splitters` | cuts long text into pieces of a set size | 4 | [python.langchain.com](https://python.langchain.com/docs/concepts/text_splitters/) |
| `fastembed` | a small free embedding model, BGE small, that runs inside this notebook: no key, no cost | 5 | [qdrant.github.io/fastembed](https://qdrant.github.io/fastembed/) |

To see every setting a library offers, run the **Look it up yourself** cell at the end, or type
`help(name)` in any cell, for example `help(pd.read_excel)`.

**Five words used throughout.**

- A **Document** is LangChain's container: `page_content` holds the text, `metadata` holds labels.
- A **chunk** is one small piece of a document. Search works on chunks, not whole files.
- **Metadata** is the label on a chunk: which file, which page, which year, current or not.
- An **embedding** is a list of numbers that stands for the meaning of a piece of text. Texts with
  similar meaning get similar lists.
- A **vector store** keeps the chunks with their embeddings and finds the ones closest to a question.

## Step 0. Install the libraries

**What this does.** Installs the libraries from the table above into this notebook's session, then
restarts Python so the new versions load. It takes about a minute. The restart is expected.

In [ ]:
# %pip installs Python libraries into this notebook's session (-q keeps the output short).
# langchain-core:            the Document object and the in-memory vector store
# langchain-community:       ready-made loaders (PDF, Word) and the FastEmbed connector
# langchain-text-splitters:  cuts long text into pieces
# pypdf, docx2txt:           the PDF and Word readers that LangChain's loaders call
# python-docx:               reads Word tables cell by cell (used to check the Word loader in Step 3b)
# python-pptx, openpyxl:     read PowerPoint slides and Excel sheets
# fastembed:                 a small free embedding model that runs inside the notebook
%pip install -q langchain-core langchain-community langchain-text-splitters pypdf docx2txt python-docx python-pptx openpyxl fastembed

In [ ]:
# Restart Python so the libraries installed above are the ones that get loaded.
# Everything defined before this line is forgotten, which is fine: nothing has run yet.
dbutils.library.restartPython()

In [ ]:
# os lets us set settings for this session; warnings controls which notices are shown.
import os, warnings

# Hide notices that are not errors, such as "langchain-community is being split into smaller packages".
warnings.filterwarnings("ignore")

# Turn off the download progress bars that the embedding model would print in Step 5.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

# Show only real errors from the model download, not routine messages.
os.environ["HF_HUB_VERBOSITY"] = "error"

# A simple signal that this cell finished.
print("ready")

## Step 1. Get the seven documents

**What this does.** Copies the seven FreshCart files from the course volume into a folder for this
session, then lists them.

**What you should see.** Seven file names: three PDFs of policies and agreements, one scanned PDF, a
Word file, a PowerPoint file and an Excel file.

In [ ]:
# The course files: the documents, the warehouse and the test questions, from this lab's GitHub
# repository. They are downloaded once and kept in one zip file, PACK_ZIP, that later steps read.
import io, pathlib, urllib.request, zipfile

PACK_ZIP = "/tmp/freshcart_course_files.zip"
if not pathlib.Path(PACK_ZIP).exists():
    repo = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen("https://github.com/mogupta84/ai-engineer-data-labs/archive/refs/heads/main.zip", timeout=120).read()))
    with zipfile.ZipFile(PACK_ZIP, "w") as out:
        for name in repo.namelist():
            inside = name.split("/", 1)[1] if "/" in name else ""     # drop the repository folder name
            if inside.startswith("data/") and not name.endswith("/"):
                out.writestr(inside, repo.read(name))
print("course files ready:", PACK_ZIP)

In [ ]:
# zipfile opens .zip archives; pathlib handles folder and file paths.
import zipfile, pathlib

# Where the course keeps all lab files, packed into one zip file.
PACK = PACK_ZIP

# A folder for this session's copy of the seven documents.
DOCS = pathlib.Path("/tmp/lab01_essentials/docs")

# Create that folder; parents=True also creates the folders above it, exist_ok=True means "fine if it is already there".
DOCS.mkdir(parents=True, exist_ok=True)

# Open the zip file. "with" closes it again automatically at the end of the block.
with zipfile.ZipFile(PACK) as z:
    # Go through every file name stored inside the zip.
    for name in z.namelist():
        # Keep only the documents in data/raw/, and skip folder entries (they end with "/").
        if name.startswith("data/raw/") and not name.endswith("/"):
            # Read the file out of the zip and write it into our folder under its plain file name.
            (DOCS / pathlib.Path(name).name).write_bytes(z.read(name))

# List what we copied, in alphabetical order.
for f in sorted(DOCS.iterdir()):
    # Print the size in bytes, right-aligned with thousands separators, then the file name.
    print(f"{f.stat().st_size:>8,} bytes   {f.name}")

## Step 2. Read the PDFs, and find the one that comes back empty

**What this does.** `PyPDFLoader` from LangChain reads each PDF and returns one `Document` per page.
The cell prints how many characters it found on each file.

**What you should see.** About 1,200 to 1,700 characters for the policies and the agreement, and **0** for
`support_escalation_sop_scanned.pdf`. That file is a scanned image: there is no text inside for the
reader to find. Notice that nothing raised an error. If you did not count, you would never know a
whole document was missing from your search.

In [ ]:
# PyPDFLoader is LangChain's PDF reader. It uses the pypdf library underneath.
from langchain_community.document_loaders import PyPDFLoader

# An empty list that will collect a Document for every PDF page.
pdf_docs = []

# Go through every file in the folder whose name ends in .pdf, in alphabetical order.
for f in sorted(DOCS.glob("*.pdf")):
    # Read the file. load() returns a list with one Document per page.
    pages = PyPDFLoader(str(f)).load()

    # Count the characters of text found on all pages; strip() ignores spaces and blank lines.
    chars = sum(len(p.page_content.strip()) for p in pages)

    # Print the file name, its number of pages and how many characters were found.
    print(f"{f.name:<40} {len(pages)} page(s)   {chars:>6,} characters")

    # Add this file's pages to the collection.
    pdf_docs += pages

# Show part of one real document, so you can see what a Document's text looks like.
print("\nthe first 300 characters of the 2024 refund policy:\n")
for d in pdf_docs:
    # metadata["source"] holds the file path the loader read; look for the 2024 policy.
    if "v3_2024" in d.metadata["source"]:
        # page_content is the text; [:300] keeps the first 300 characters.
        print(d.page_content[:300])
        # Stop after the first match.
        break

## Step 3. Read the Word, PowerPoint and Excel files

**What this does.** Each format needs a different reader, but every result becomes the same thing: a
LangChain `Document` with text and a `source` label.

- **Word:** LangChain's `Docx2txtLoader`.
- **PowerPoint:** LangChain's own PowerPoint loader needs a large extra package, so this cell builds
  the `Document`s itself with `python-pptx`, one per slide. This is all a loader does inside.
- **Excel:** `pandas` reads the sheet into a table, and the table becomes one `Document` as text.

**What you should see.** Word: 1 Document, 991 characters. PowerPoint: 3 Documents, one per slide,
656 characters. Excel: 1 Document, 2,249 characters. Then the total from all four formats: 9 Documents.

In [ ]:
# pandas reads Excel sheets into a table (it uses openpyxl underneath).
import pandas as pd

# Presentation opens a PowerPoint file (from the python-pptx library).
from pptx import Presentation

# Docx2txtLoader is LangChain's Word reader. It uses the docx2txt library underneath.
from langchain_community.document_loaders import Docx2txtLoader

# Document is LangChain's container: page_content for the text, metadata for the labels.
from langchain_core.documents import Document

# WORD: one call reads the whole file into a list with one Document.
word_docs = Docx2txtLoader(str(DOCS / "support_sla_matrix.docx")).load()

# POWERPOINT: we build one Document per slide ourselves.
slide_docs = []
pptx_file = DOCS / "support_qbr_q1_2024.pptx"
# enumerate gives a counter (0, 1, 2, ...) alongside each slide.
for number, slide in enumerate(Presentation(pptx_file).slides):
    # Collect the text of every shape on the slide that holds text (titles, text boxes).
    texts = [shape.text_frame.text for shape in slide.shapes if shape.has_text_frame]
    # Wrap that text in a Document, labelled with the file and the slide number (counting from 1).
    slide_docs.append(Document(page_content="\n".join(texts),
                               metadata={"source": str(pptx_file), "slide": number + 1}))

# EXCEL: read the first sheet into a table.
table = pd.read_excel(DOCS / "refund_claims_march_2024.xlsx")
# Turn the whole table into plain text (index=False leaves out the row numbers) and wrap it in one Document.
excel_docs = [Document(page_content=table.to_string(index=False),
                       metadata={"source": str(DOCS / "refund_claims_march_2024.xlsx")})]

# Print, for each format, how many Documents it gave and how many characters of text they hold.
for name, docs in [("Word", word_docs), ("PowerPoint", slide_docs), ("Excel", excel_docs)]:
    print(f"{name:<11} {len(docs)} Document(s)   {sum(len(d.page_content) for d in docs):>6,} characters")

# Put the Documents from all four formats into one list.
all_docs = pdf_docs + word_docs + slide_docs + excel_docs
print(f"\nall four formats: {len(all_docs)} Documents")

### Step 3b. Check: the Word table lost its column names

**What this does.** The Word file holds one table: a row of column names, then one row per priority.
The cell prints what LangChain's Word loader returned, then reads the same table cell by cell with
`python-docx` to show the column names that are in the file.

**What you should see.** The loader's text is a run of values such as `15 minutes  2 hours  Supervisor
immediately`, with no column names. Is 15 minutes the first response time or the resolution time?
You cannot tell. The table's first row says: Priority, Trigger, First response TAT, Resolution TAT,
Escalation path. A search can find this text, but nobody can answer a TAT question from it correctly.

In [ ]:
# python-docx opens a Word file and gives access to its tables, rows and cells.
import docx

# What LangChain's Word loader returned. The table comes after a few lines of introduction, so
# find where "15 minutes" appears and print 200 characters from just before it.
loader_text = word_docs[0].page_content.replace("\n", "  ")
start = loader_text.find("15 minutes") - 70
print("WHAT THE LOADER GAVE US, WHERE THE TABLE IS:")
print(" ", loader_text[start:start + 200], "\n")

# Open the same file with python-docx and take its first table.
word_table = docx.Document(DOCS / "support_sla_matrix.docx").tables[0]

# The first row of the table holds the column names.
print("WHAT THE TABLE'S FIRST ROW SAYS:")
print(" ", [cell.text for cell in word_table.rows[0].cells])

# The second row, cell by cell, so each value sits under its column name.
print("\nTHE SECOND ROW, WITH ITS COLUMN NAMES:")
for name, cell in zip(word_table.rows[0].cells, word_table.rows[1].cells):
    print(f"  {name.text:<22} {cell.text}")

## Step 4. Cut the documents into pieces, and label every piece

**What this does.** `RecursiveCharacterTextSplitter` cuts each document into chunks of at most 500
characters. Neighbouring chunks share 50 characters, so text right at a cut appears in both. Then every
chunk gets three labels in its `metadata`: the file name, the year of the document, and whether it is
the **current** version. The 2022 refund policy was replaced by the 2024 one, so it is labelled
`current: False`.

**What you should see.** 21 chunks from the 9 Documents, and one chunk from the 2024 policy printed
with its three labels.

In [ ]:
# RecursiveCharacterTextSplitter cuts text at paragraph breaks first, then lines, then words.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Pieces of at most 500 characters; neighbouring pieces share 50 characters, so a sentence cut at
# the edge of one piece still appears whole in the next.
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

# Cut every Document into chunks. Each chunk is again a Document, and it keeps the original labels.
# The "if" drops chunks that are empty, such as the one from the scanned PDF.
chunks = [c for c in splitter.split_documents(all_docs) if c.page_content.strip()]

# Add our own labels to every chunk.
for c in chunks:
    # The file name without the folder path, e.g. "refund_policy_v3_2024.pdf".
    name = pathlib.Path(c.metadata["source"]).name
    c.metadata["file"] = name
    # 2022 for the old refund policy; every other document is from 2024.
    c.metadata["year"] = 2022 if "2022" in name else 2024
    # False only for the 2022 policy, because the 2024 policy replaced it.
    c.metadata["current"] = name != "refund_policy_v2_2022.pdf"

print(len(chunks), "chunks from", len(all_docs), "Documents\n")

# Show one chunk from the 2024 policy, with its text and its labels.
for c in chunks:
    if c.metadata["file"] == "refund_policy_v3_2024.pdf":
        # The first 200 characters, with line breaks turned into spaces so it prints on one line.
        print("text  :", c.page_content[:200].replace("\n", " "), "...")
        # Only the three labels we added, not every label the loader attached.
        print("labels:", {k: c.metadata[k] for k in ("file", "year", "current")})
        break

### Step 4b. Check: Excel pieces cut away from their column names

**What this does.** The Excel sheet became one long block of text in Step 3, and the splitter cut it
into 5 pieces of at most 500 characters. The cell says which pieces contain the column names, then
prints the first piece and the third piece in full.

**What you should see.** Two problems. First, the sheet starts with a title and three note rows, and
the real column names (Claim ID, Order ID, Customer ID, Reason, Claim date, Amount (INR), Status) are
on row 5. `pd.read_excel` took the title as the header, which is why the first piece shows
`Unnamed: 1`, `Unnamed: 2` and `NaN` (empty) cells: it is almost all title and padding. Second, the
real column names land in piece 2 only. Pieces 3, 4 and 5 are rows of values with no names at all.
A search can return piece 3, and nobody reading it knows what 1290, 355 or "Declined" refer to.
The fix is to tell pandas where the header is, `pd.read_excel(..., header=4)`, and to repeat the
column names in every piece.

In [ ]:
# Keep only the chunks that came from the Excel file, in the order they were cut.
excel_chunks = [c for c in chunks if c.metadata["file"] == "refund_claims_march_2024.xlsx"]
print(len(excel_chunks), "pieces came from the Excel sheet\n")

# For each piece, does it contain the real column names? enumerate counts the pieces from 1.
for number, piece in enumerate(excel_chunks, start=1):
    print(f"piece {number}: column names inside? {'Claim ID' in piece.page_content}")

# The first piece in full: the title read as a header ("Unnamed: 1" ...) and empty cells ("NaN").
print("\nPIECE 1:")
print(excel_chunks[0].page_content, "\n")

# The third piece in full: rows of values, and no line telling you what each column is.
print("PIECE 3:")
print(excel_chunks[2].page_content)

## Step 5. Turn text into numbers (embeddings)

**What this does.** `FastEmbedEmbeddings` loads BGE small, a free embedding model, inside this notebook.
The first run downloads it, about 130 MB. It turns any text into a list of 384 numbers. Two texts
with similar meaning get lists that point in a similar direction.

**What you should see.** The length of one embedding, 384, and its first five numbers, then a small
test: a question about refund limits scores about 0.84 against a refund sentence and about 0.53
against a delivery-time sentence.

In [ ]:
# FastEmbedEmbeddings connects LangChain to the fastembed library, which runs the model locally.
from langchain_community.embeddings import FastEmbedEmbeddings

# numpy does arithmetic on lists of numbers.
import numpy as np

# Load BGE small, a free embedding model. The first run downloads it (about 130 MB).
embedder = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# Turn one question into its embedding: a list of numbers that stands for its meaning.
v = embedder.embed_query("How many refund claims can a customer make?")
# How many numbers there are, and the first five, rounded to 3 decimals.
print("numbers per embedding:", len(v), "  first five:", np.round(v[:5], 3))

def similarity(a, b):
    """Cosine similarity of two texts: near 1 means very similar meaning, near 0 means unrelated."""
    # Embed both texts and turn the results into numpy arrays.
    a, b = np.array(embedder.embed_query(a)), np.array(embedder.embed_query(b))
    # Multiply the two lists number by number and add up (a @ b), then divide by both lengths.
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

# Compare one question with a related sentence and with an unrelated one.
q = "How many refund claims can a customer make?"
print("\nquestion vs a refund sentence  :", round(similarity(q, "Customers may submit up to three refund claims a month."), 3))
print("question vs a delivery sentence:", round(similarity(q, "Orders are delivered within forty minutes."), 3))

## Step 6. Store the chunks and search them: watch the wrong policy win

**What this does.** `InMemoryVectorStore` embeds every chunk and keeps them in memory. Then
`similarity_search_with_score` takes a question, embeds it, and returns the closest chunks with their
similarity score.

**What you should see.** The question is "How many working days until an approved refund is
credited?" The top result, 0.877, is the **2022** policy: "seven to ten working days". That policy was
replaced. The current 2024 policy says **five to seven** working days, and it only comes second, 0.791.
The search did its job: the 2022 text really is the closest in meaning. It just has no way of knowing
that the 2022 policy is out of date.

In [ ]:
# InMemoryVectorStore keeps chunks and their embeddings in this notebook's memory.
from langchain_core.vectorstores import InMemoryVectorStore

# Embed every chunk with our model and store chunk plus embedding together.
store = InMemoryVectorStore.from_documents(chunks, embedder)
print(len(chunks), "chunks stored\n")

# The question we will search with.
question = "How many working days until an approved refund is credited?"

# Embed the question, compare it with every stored chunk, and return the 3 closest with their score.
for doc, score in store.similarity_search_with_score(question, k=3):
    # A short name for the chunk's labels.
    m = doc.metadata
    # Print the score (3 decimals), the file, and the two labels that matter here.
    print(f"{score:.3f}  {m['file']:<32} year {m['year']}  current {m['current']}")
    # Print the chunk's full text on one line, indented.
    print("       ", doc.page_content.replace("\n", " "), "\n")

## Step 7. Search again, using the labels

**What this does.** The same search with a `filter`: only chunks whose `current` label is `True` may be
returned. The labels from Step 4 are what make this one line possible.

**What you should see.** The 2022 policy is gone. The top result is now the 2024 policy, 0.791, with
"five to seven working days". Same question, same model, same scores: only the label made the
difference.

**Try this.** In Step 4, change `chunk_size=500` to `300` and run Steps 4 to 7 again. The 2024 policy
now wins without the filter, 0.846, but the 2022 policy is right behind it at 0.813. How you cut the
text changes what the search finds, which is why the full lab measures it.

In [ ]:
# A filter is a small function: given a chunk, return True to allow it, False to skip it.
# This one allows only chunks whose "current" label is True.
only_current = lambda doc: doc.metadata["current"]

# The same search as Step 6, with the filter applied before the 3 closest are picked.
for doc, score in store.similarity_search_with_score(question, k=3, filter=only_current):
    m = doc.metadata
    print(f"{score:.3f}  {m['file']:<32} year {m['year']}  current {m['current']}")
    print("       ", doc.page_content.replace("\n", " "), "\n")

### Step 7b. Check: a question the documents cannot answer still gets results

**What this does.** Asks a question no FreshCart document covers: refunds for grocery orders in
Singapore. FreshCart's policy covers food delivery in India only.

**What you should see.** Three results with confident scores. The top one, about 0.82, is the scope
section of the 2024 refund policy, which says it covers food delivery in India. That score is
*higher* than the 0.79 of the refund question in Step 7, which the documents do answer. A similarity
score measures closeness of meaning, not whether the answer is there, so no cut-off on the score can
separate the two. Nothing in the search says "not found". Lab 4 builds an answer service that has to
deal with exactly this.

In [ ]:
# A question the seven documents cannot answer.
no_answer = "What is FreshCart's refund policy for grocery orders in Singapore?"

# The same search as Step 7, with the same filter.
for doc, score in store.similarity_search_with_score(no_answer, k=3, filter=only_current):
    m = doc.metadata
    print(f"{score:.3f}  {m['file']:<32}")
    # The first 160 characters of each result are enough to see it is not about Singapore or groceries.
    print("       ", doc.page_content[:160].replace("\n", " "), "...\n")

# For comparison: the top score for the refund question in Step 7, which the documents do answer.
answered = store.similarity_search_with_score(question, k=1, filter=only_current)[0][1]
print(f"top score for the question the documents DO answer: {answered:.3f}")

## What you found

Five problems, and not one of them raised an error.

1. **A whole document can disappear.** The scanned PDF gave 0 characters (Step 2). Always count
   what each loader returns. The fix is a reader that can see images: the full Lab 1 uses one.
2. **A Word table can lose its column names** (Step 3b). The values survive, but not what they
   mean. The fix is a loader that keeps the table structure, or writing each row with its column names.
3. **Excel needs to be told where the header is, and pieces lose it** (Step 4b). The fix is
   `header=` in `read_excel`, and putting the column names in every piece.
4. **The best match in meaning can be the wrong answer** (Step 6). Similarity cannot know a policy
   was replaced. A label added when loading, and a filter at search time, fix it (Step 7).
5. **Search always returns something** (Step 7b). A question with no answer scored higher than one
   with an answer. Deciding when to say "not found" is Lab 4.

**Where this goes next.** The full Lab 1 notebook, `lab01_ingest`, does the same steps with more
formats checked, three loader libraries compared, and five real support questions scored.

## Look it up yourself: every setting the libraries offer

**What this does.** For the eight things this lab calls, the cell prints three lines: every parameter
it accepts (with its default value, where it has one), the values this lab passed, and one setting
worth trying. A **parameter** is a named input, such as `chunk_size`; the value you pass for it is the
**argument**, such as `500`. The list is read from the installed library, so it is always current.

**What you should see.** Eight short blocks. Notice how many settings each one has, and that this lab
changed only a few. `header` in `pd.read_excel` is the fix for Step 4b. And `filter`, which Step 7
uses, is missing from the search's list: some settings arrive through a catch-all input called
`**kwargs`, and only `help()` or the docs tell you they exist.

For the full description of any of them, run `help(...)`, for example
`help(RecursiveCharacterTextSplitter)`, or follow the Docs link in the table at the top.

In [ ]:
# inspect reads the list of parameters a function or class accepts; textwrap folds long lines.
import inspect, textwrap
# python-docx, imported here as docx, so we can look up docx.Document from Step 3b.
import docx

# parameters(thing): the parameters `thing` accepts, as text such as "chunk_size=4000".
# Some classes pass most settings on to a parent class through **kwargs, so it looks there too.
def parameters(thing):
    found = []
    # Some classes (FastEmbedEmbeddings is one) list their settings as "fields", not on __init__.
    if hasattr(thing, "model_fields"):
        for name, field in thing.model_fields.items():
            # A field with no default must be given; show just its name then.
            found.append(name if field.is_required() else f"{name}={field.default!r}")
        return found
    # A class keeps its parameters on __init__, and maybe on its parents' __init__ as well.
    if inspect.isclass(thing):
        places = [c.__init__ for c in thing.__mro__ if "__init__" in vars(c)]
    else:
        places = [thing]
    for place in places:
        try:
            sig = inspect.signature(place)
        except (TypeError, ValueError):       # some built-in objects cannot be read; skip them
            continue
        for p in sig.parameters.values():
            # Skip "self", and the *args / **kwargs catch-alls, which are not named settings.
            if p.name == "self" or p.kind in (p.VAR_POSITIONAL, p.VAR_KEYWORD):
                continue
            # Show "name=default" when there is a default, or just "name" when you must give it.
            text = p.name if p.default is p.empty else f"{p.name}={p.default!r}"
            # Add it only once, even if a parent class lists the same name again.
            if p.name not in [f.split("=")[0] for f in found]:
                found.append(text)
        # Stop climbing to parent classes once a level has no **kwargs to pass settings on.
        if not any(p.kind == p.VAR_KEYWORD for p in sig.parameters.values()):
            break
    return found

# look_up prints the three lines for one library function or class.
def look_up(name, thing, this_lab_passed, worth_a_look):
    # name is how this lab writes it, for example "pd.read_excel".
    print(name)
    # textwrap.fill folds the long list at 110 characters and lines the folds up under the list.
    print(textwrap.fill("every parameter : " + ", ".join(parameters(thing)), 110, subsequent_indent=" " * 18))
    print("this lab passed :", this_lab_passed)
    print("worth a look    :", worth_a_look, "\n")

# One call per thing this lab used: the thing itself, the values we passed, one setting to try.
look_up("PyPDFLoader", PyPDFLoader, "file_path = the PDF's location, passed without its name",
        "mode: one Document per page, or one for the whole file")
look_up("Docx2txtLoader", Docx2txtLoader, "file_path = the Word file's location",
        "almost nothing to set, which is why Step 3b lost the column names")
look_up("pd.read_excel", pd.read_excel, "io = the Excel file's location",
        "header: which row holds the column names; header=4 fixes Step 4b")
look_up("Presentation (python-pptx)", Presentation, "pptx = the PowerPoint file's location",
        "nothing to set here: the detail is in slide.shapes and shape.text_frame")
look_up("docx.Document (python-docx)", docx.Document, "docx = the Word file's location",
        "nothing to set here: tables, rows and cells are reached after opening")
look_up("RecursiveCharacterTextSplitter", RecursiveCharacterTextSplitter, "chunk_size = 500, chunk_overlap = 50",
        "separators: the places it tries to cut, in order (paragraph, line, space)")
look_up("FastEmbedEmbeddings", FastEmbedEmbeddings, 'model_name = "BAAI/bge-small-en-v1.5", the same as the default, written out so you can see which model',
        "batch_size: how many pieces it turns into numbers at once")
look_up("store.similarity_search_with_score", InMemoryVectorStore.similarity_search_with_score, "query = the question, k = 3, filter = only_current",
        "filter is not in the list above: it arrives through a catch-all input (**kwargs), so only help() shows it")

## Your notes

Double click to type: what surprised you, and one question you still have.